<a href="https://colab.research.google.com/github/viviantram03/labb-1/blob/main/lab4_without_pipieline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pyspark

!wget -q -O bank.csv https://raw.githubusercontent.com/selva86/datasets/master/bank-full.csv

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("BankMarketing_Without_Pipeline").getOrCreate()

print("Spark Version: ", spark.version)

Spark Version:  4.0.4


In [ ]:
df = spark.read.format("csv").option("header", "true").option("sep", ";").option("inferSchema", "true").load("bank.csv")

df.printSchema()
df.show(5)

root
 |-- age: integer (nullable = true)
 |-- job: string (nullable = true)
 |-- marital: string (nullable = true)
 |-- education: string (nullable = true)
 |-- default: string (nullable = true)
 |-- housing: string (nullable = true)
 |-- loan: string (nullable = true)
 |-- contact: string (nullable = true)
 |-- month: string (nullable = true)
 |-- day_of_week: string (nullable = true)
 |-- duration: integer (nullable = true)
 |-- campaign: integer (nullable = true)
 |-- pdays: integer (nullable = true)
 |-- previous: integer (nullable = true)
 |-- poutcome: string (nullable = true)
 |-- emp.var.rate: double (nullable = true)
 |-- cons.price.idx: double (nullable = true)
 |-- cons.conf.idx: double (nullable = true)
 |-- euribor3m: double (nullable = true)
 |-- nr.employed: double (nullable = true)
 |-- y: string (nullable = true)

+---+---------+-------+-----------+-------+-------+----+---------+-----+-----------+--------+--------+-----+--------+-----------+------------+--------------+

In [ ]:
from pyspark.sql.functions import col, isnull, when, count

dataset = df.select(
    col("age").cast("float"),
    col("job"),
    col("marital"),
    col("education"),
    col("duration").cast("float"),
    col("campaign").cast("float"),
    col("y"),
)

dataset.select([count(when(isnull(c), c)).alias(c) for c in dataset.columns]).show()

dataset = dataset.dropna(how="any")
dataset.show(5)


+---+---+-------+---------+--------+--------+---+
|age|job|marital|education|duration|campaign|  y|
+---+---+-------+---------+--------+--------+---+
|  0|  0|      0|        0|       0|       0|  0|
+---+---+-------+---------+--------+--------+---+

+----+---------+-------+-----------+--------+--------+---+
| age|      job|marital|  education|duration|campaign|  y|
+----+---------+-------+-----------+--------+--------+---+
|56.0|housemaid|married|   basic.4y|   261.0|     1.0| no|
|57.0| services|married|high.school|   149.0|     1.0| no|
|37.0| services|married|high.school|   226.0|     1.0| no|
|40.0|   admin.|married|   basic.6y|   151.0|     1.0| no|
|56.0| services|married|high.school|   307.0|     1.0| no|
+----+---------+-------+-----------+--------+--------+---+
only showing top 5 rows


In [ ]:
from pyspark.ml.feature import StringIndexer, VectorAssembler

dataset = StringIndexer(inputCol="job", outputCol="job_index", handleInvalid="keep").fit(dataset).transform(dataset)
dataset = StringIndexer(inputCol="marital", outputCol="marital_index", handleInvalid="keep").fit(dataset).transform(dataset)
dataset = StringIndexer(inputCol="education", outputCol="education_index", handleInvalid="keep").fit(dataset).transform(dataset)

dataset = StringIndexer(inputCol="y", outputCol="label", handleInvalid="keep").fit(dataset).transform(dataset)

dataset = dataset.drop("job", "marital", "education", "y")
dataset.show(5)

+----+--------+--------+---------+-------------+---------------+-----+
| age|duration|campaign|job_index|marital_index|education_index|label|
+----+--------+--------+---------+-------------+---------------+-----+
|56.0|   261.0|     1.0|      8.0|          0.0|            4.0|  0.0|
|57.0|   149.0|     1.0|      3.0|          0.0|            1.0|  0.0|
|37.0|   226.0|     1.0|      3.0|          0.0|            1.0|  0.0|
|40.0|   151.0|     1.0|      0.0|          0.0|            5.0|  0.0|
|56.0|   307.0|     1.0|      3.0|          0.0|            1.0|  0.0|
+----+--------+--------+---------+-------------+---------------+-----+
only showing top 5 rows


In [ ]:
feature_cols = ["age", "duration", "campaign", "job_index", "marital_index", "education_index"]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

transformed_data = assembler.transform(dataset)
transformed_data.select("features", "label").show(5, truncate=False)

+----------------------------+-----+
|features                    |label|
+----------------------------+-----+
|[56.0,261.0,1.0,8.0,0.0,4.0]|0.0  |
|[57.0,149.0,1.0,3.0,0.0,1.0]|0.0  |
|[37.0,226.0,1.0,3.0,0.0,1.0]|0.0  |
|[40.0,151.0,1.0,0.0,0.0,5.0]|0.0  |
|[56.0,307.0,1.0,3.0,0.0,1.0]|0.0  |
+----------------------------+-----+
only showing top 5 rows


In [ ]:
(training_data, test_data) = transformed_data.randomSplit([0.8, 0.2], seed=42)

print("Training Dataset Count: ", training_data.count())
print("Test Dataset Count: ", test_data.count())

Training Dataset Count:  32977
Test Dataset Count:  8211


In [ ]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(labelCol="label", featuresCol="features", maxDepth=5, numTrees=20,)
model = rf.fit(training_data)

predictions_training = model.transform(training_data)
predictions_test = model.transform(test_data)

predictions_test.select("label", "prediction", "probability").show(5)


+-----+----------+--------------------+
|label|prediction|         probability|
+-----+----------+--------------------+
|  0.0|       0.0|[0.88142578147730...|
|  0.0|       0.0|[0.89580508140682...|
|  0.0|       0.0|[0.89580508140682...|
|  1.0|       0.0|[0.87860284940183...|
|  1.0|       0.0|[0.71434358499177...|
+-----+----------+--------------------+
only showing top 5 rows


In [ ]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator_acc = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)
train_acc = evaluator_acc.evaluate(predictions_training)
test_acc = evaluator_acc.evaluate(predictions_test)

evaluator_f1 = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1")
train_f1 = evaluator_f1.evaluate(predictions_training)
test_f1 = evaluator_f1.evaluate(predictions_test)

print("=== Results without Pipeline ===")
print("Training Accuracy: ", train_acc)
print("Test Accuracy: ", test_acc)
print("Training F1-Score: ", train_f1)
print("Test F1-Score: ", test_f1)


=== Results without Pipeline ===
Training Accuracy:  0.8970494587136489
Test Accuracy:  0.8906345146754354
Training F1-Score:  0.8704199194041197
Test F1-Score:  0.8622065854173054
